# Part 1.2e — Inverse-Distance-Weighted, Connected k-NN Graph

This builds on `03c` with two changes aimed at producing **one continuous manifold**
instead of the 66 fragments seen there, so the Laplacian Eigenmap can diffuse features
across the whole dataset:

1. **Inverse-distance edge weights.** Each edge is weighted by `1 / haversine_km`, so
   nearby blocks connect strongly and distant ones weakly. The embedding then models
   the **flow of housing value across geographic distance** rather than treating all
   in-radius neighbours as equal.
2. **A larger radius that connects the graph.** Because far neighbours are now
   down-weighted, a bigger `RADIUS_WALK_KM` is safe — it bridges the fragmentation
   without letting long edges dominate. Tune `K_NEIGHBORS` / `RADIUS_WALK_KM` until the
   reported **connected-component count reaches 1**.

The eigenmap extraction is also made **robust to residual disconnection**: it skips
*all* null-space (zero-eigenvalue) modes and returns the first `k` with `λ > 0` — so
even a few leftover components no longer yield the degenerate `[-0,-0,-0]` output.

This notebook:
1. Loads the block and MRT datasets.
2. Builds the inverse-distance-weighted, radius-capped k-NN graph (union-symmetrised).
3. Plots the graph (blue) with the MRT network (red).
4. Runs the Laplacian Eigenmap spectral decomposition (robust extraction).
5. Extends the embedding to test houses via a weighted Nyström kernel.
6. Evaluates the embedding's information value against `MONTHLY_RENT`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.sparse import triu, csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors

# Haversine distances come back in radians on the unit sphere; scale by Earth radius for km.
EARTH_RADIUS_KM = 6371.0088

# Walking paths detour around roads/blocks, so walking distance exceeds straight-line
# (Haversine) distance. Divide the walking cutoff by this factor to get the Haversine radius.
DETOUR_FACTOR = 1.35

# Approximate lat/long bounding box of mainland Singapore
SG_LAT_RANGE = (1.20, 1.48)
SG_LON_RANGE = (103.60, 104.10)

## 1. Load the house and MRT datasets

In [2]:
# TRAIN_HOUSE_PATH = "graph_feature_store/train-house.csv"
TRAIN_HOUSE_PATH = "/Users/MONSG0140/Desktop/projects/others/graph_regression_model/raw_graph_data/sg-hdb-block.csv"
STATION_KEYS_PATH = "graph_feature_store/station-keys.csv"
MRT_EDGES_PATH = "raw_graph_data/sg-mrt-edges-weighted.csv"

train_house = pd.read_csv(TRAIN_HOUSE_PATH)
stations = pd.read_csv(STATION_KEYS_PATH)
mrt_edges = pd.read_csv(MRT_EDGES_PATH)

print(f"Houses   : {len(train_house):,}")
print(f"Stations : {len(stations)}")
print(f"MRT edges: {len(mrt_edges)}")

Houses   : 9,464
Stations : 192
MRT edges: 224


In [8]:
train_house['BLOCK'].drop_duplicates().count()

2735

In [6]:
train_house['TOWN'].drop_duplicates().count()

26

## 2. Build the inverse-distance-weighted, radius-capped k-NN graph

Same radius-capped k-NN construction as `03c` (top-`K` within a walking radius,
union-symmetrised), with two changes:

1. **Inverse-distance edge weights** — each edge weight is `1 / haversine_km`, so
   nearby blocks are strongly connected and distant ones weakly (a 1 m floor guards
   against division by zero). This penalises distance and lets the Laplacian model the
   flow of value across space.
2. **A larger radius to connect the graph** — since far neighbours are down-weighted,
   a bigger `RADIUS_WALK_KM` bridges the `03c` fragmentation without long edges
   dominating.

Union symmetrisation is unchanged; Haversine distance is symmetric, so the two edge
directions carry the same weight and `max(A, Aᵀ)` simply keeps it. Tune the parameters
until **connected components = 1**.

In [ ]:
# --- Tunable parameters -------------------------------------------------
K_NEIGHBORS = 10           # keep up to this many nearest neighbours per block
RADIUS_WALK_KM = 5         # ...within this WALKING distance (km); larger is safe now
#                            because edges are inverse-distance weighted
# ------------------------------------------------------------------------

radius_hav_km = RADIUS_WALK_KM / DETOUR_FACTOR      # straight-line equivalent
radius_rad = radius_hav_km / EARTH_RADIUS_KM

coords_rad = np.radians(train_house[["LATITUDE", "LONGITUDE"]].to_numpy())
n = coords_rad.shape[0]

# k nearest neighbours (request K+1 to absorb the self-match at distance 0)
nn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric="haversine", algorithm="ball_tree")
nn.fit(coords_rad)
knn_dist, knn_idx = nn.kneighbors(coords_rad)

# Keep entries that are (a) not self and (b) within the radius -> top-k WITHIN radius
row_ids = np.arange(n)[:, None]
not_self = knn_idx != row_ids
keep = not_self & (knn_dist <= radius_rad)
rows = np.broadcast_to(row_ids, knn_idx.shape)[keep]
cols = knn_idx[keep]

# INVERSE-DISTANCE edge weights (km): closer blocks -> stronger edges.
dist_km = knn_dist[keep] * EARTH_RADIUS_KM
w = 1.0 / np.maximum(dist_km, 1e-3)                 # floor at 1 m to avoid div-by-zero
A_dir = csr_matrix((w, (rows, cols)), shape=(n, n))
A = A_dir.maximum(A_dir.T)                          # union; symmetric weights so max keeps it

unweighted_deg = np.asarray((A > 0).sum(axis=1)).ravel()
n_edges = int(A.nnz // 2)                           # symmetric matrix double-counts each edge
nearest_nonself = np.where(not_self, knn_dist, np.inf).min(axis=1)
n_isolated = int((nearest_nonself > radius_rad).sum())
n_comp = connected_components(A, directed=False)[0]

print(f"K_NEIGHBORS = {K_NEIGHBORS}, RADIUS_WALK_KM = {RADIUS_WALK_KM} km "
      f"(Haversine {radius_hav_km:.3f} km)")
print(f"Edges (union)      : {n_edges:,}")
print(f"Mean degree        : {unweighted_deg.mean():.2f}  (edges per node, unweighted)")
print(f"Edge weight range  : {w.min():.3f} – {w.max():.3f}  (1/km)")
print(f"Isolated blocks    : {n_isolated:,} ({n_isolated / n:.2%})")
print(f"Connected components: {n_comp}   <- tune K / radius until this is 1")

## 3. Plot the house graph (blue) with the MRT graph (red)

House nodes and their radius-capped k-NN edges are drawn in **blue**; MRT stations
and their track edges in **red**, matching the DBSCAN notebook's overlay.

This graph is far sparser than the raw epsilon ball (~`K` edges per house rather than
hundreds), so it should render without trouble. The `max_display_edges` cap is kept
as a safety valve and prints a note if it ever samples.

In [4]:
def _edge_segments(lon, lat, ij_pairs):
    """Flatten (i, j) index pairs into None-separated x/y arrays for a single line trace."""
    xs, ys = [], []
    for i, j in ij_pairs:
        xs += [lon[i], lon[j], None]
        ys += [lat[i], lat[j], None]
    return xs, ys


def plot_house_graph(
    house_df,
    house_adj,
    stations,
    mrt_edges,
    title="House graph with MRT network",
    lat_range=SG_LAT_RANGE,
    lon_range=SG_LON_RANGE,
    max_display_edges=50_000,
    seed=0,
):
    """Plot houses + their graph edges (blue) with MRT stations + edges (red).

    house_adj : sparse symmetric adjacency over the rows of house_df.
    max_display_edges : cap on house edges drawn; sampled at random if exceeded.
    """
    h_lon = house_df["LONGITUDE"].to_numpy()
    h_lat = house_df["LATITUDE"].to_numpy()

    # Unique undirected house edges from the upper triangle
    upper = triu(house_adj, k=1).tocoo()
    rows, cols = upper.row, upper.col
    n_total = len(rows)
    if n_total > max_display_edges:
        rng = np.random.default_rng(seed)
        pick = rng.choice(n_total, size=max_display_edges, replace=False)
        rows, cols = rows[pick], cols[pick]
        print(
            f"Drawing {max_display_edges:,} of {n_total:,} house edges (random sample) "
            f"to keep the plot responsive."
        )
    hx, hy = _edge_segments(h_lon, h_lat, zip(rows, cols))

    fig = px.scatter(
        title=title,
        labels={"x": "Longitude", "y": "Latitude"},
    )

    # House edges (blue), then house nodes (blue) on top
    fig.add_scatter(x=hx, y=hy, mode="lines", line=dict(color="royalblue", width=0.5),
                    opacity=0.4, name="house edges")
    fig.add_scatter(x=h_lon, y=h_lat, mode="markers",
                    marker=dict(color="royalblue", size=4), opacity=0.6, name="houses")

    # MRT edges (red), then stations (red) on top
    m_lon = stations["longitude"].to_numpy()
    m_lat = stations["latitude"].to_numpy()
    mrt_pairs, missing = [], 0
    name_to_pos = {name: k for k, name in enumerate(stations["NAME"])}
    for src, dst in zip(mrt_edges["src_station"], mrt_edges["dst_station"]):
        if src in name_to_pos and dst in name_to_pos:
            mrt_pairs.append((name_to_pos[src], name_to_pos[dst]))
        else:
            missing += 1
    if missing:
        print(f"{missing} MRT edge(s) skipped — station name not found in station-keys")
    mx, my = _edge_segments(m_lon, m_lat, mrt_pairs)

    fig.add_scatter(x=mx, y=my, mode="lines", line=dict(color="red", width=1),
                    opacity=0.8, name="MRT lines")
    fig.add_scatter(x=m_lon, y=m_lat, mode="markers", marker=dict(color="red", size=6),
                    name="MRT stations", text=stations["NAME"], hoverinfo="text")

    fig.update_xaxes(range=list(lon_range))
    fig.update_yaxes(range=list(lat_range))
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_layout(width=950, height=650, legend_title_text="")
    return fig


fig = plot_house_graph(
    train_house,
    A,
    stations,
    mrt_edges,
    title=f"Radius-capped k-NN house graph (K={K_NEIGHBORS}, "
          f"RADIUS_WALK_KM={RADIUS_WALK_KM}) with MRT network",
)
fig.show()

## 4. Laplacian Eigenmap — spectral decomposition (robust extraction)

1. **Adjacency** — the inverse-distance-weighted `A` from Section 2. `normalized_laplacian`
   handles real-valued weights (degree = row-sum of weights).
2. **Normalised Laplacian** — `L = I - D^{-1/2} A D^{-1/2}`, decomposed via the largest
   eigenvalues of `S = D^{-1/2} A D^{-1/2}` (= smallest of `L`) using `eigsh(which='LA')`,
   which also yields the affinity eigenvalues `μ_k` and degrees for Nyström.
3. **Robust non-trivial extraction.** A graph with `c` components has `c` zero
   eigenvalues. Rather than assuming `c = 1` and dropping a single mode, we request
   `c + k` modes and drop **all** with `λ ≤ tol`, keeping the first `k` with `λ > 0`.
   This returns genuine smooth coordinates even if a few components remain (they no
   longer leak in as the degenerate null-space modes seen in `03c`/`03d`).

> On a fully connected graph this is identical to "drop the one trivial mode." On a
> disconnected graph the first `λ > 0` modes localise on the largest components, so a
> single component (Section 2) is still the goal for a coherent island-wide embedding.

In [ ]:
from scipy.sparse import diags, eye
from scipy.sparse.linalg import eigsh


def normalized_laplacian(adj):
    """Symmetric normalised Laplacian  L = I - D^{-1/2} A D^{-1/2}.

    Works for weighted `adj` (degree = row-sum of weights). Returns (L, S, deg) where
    S = D^{-1/2} A D^{-1/2}. Isolated nodes (degree 0) get a 0 row in S.
    """
    adj = adj.maximum(adj.T).astype(float)          # enforce exact symmetry
    deg = np.asarray(adj.sum(axis=1)).ravel()
    d_inv_sqrt = np.zeros_like(deg)
    nz = deg > 0
    d_inv_sqrt[nz] = 1.0 / np.sqrt(deg[nz])
    D_inv_sqrt = diags(d_inv_sqrt)
    S = (D_inv_sqrt @ adj @ D_inv_sqrt).tocsr()
    L = (eye(adj.shape[0], format="csr") - S).tocsr()
    return L, S, deg


def laplacian_eigenmaps(adj, k, tol=1e-8):
    """First k NON-TRIVIAL eigenvectors of the symmetric normalised Laplacian.

    Robust to disconnection: a graph with c components has c zero eigenvalues, so we
    request c + k modes and drop ALL with lambda <= tol, keeping the first k with
    lambda > tol. Solved via the largest eigenvalues of S = D^{-1/2} A D^{-1/2}
    (== smallest of L = I - S) for numerical stability.

    Returns:
      V   : (n, k) eigenvector matrix — each ROW is a node's k-dim embedding
      lam : (k,)  Laplacian eigenvalues, ascending  (lam = 1 - mu, all > tol)
      mu  : (k,)  normalised-affinity eigenvalues    (reused by Nystrom)
      deg : (n,)  node degrees                        (reused by Nystrom)
    """
    _, S, deg = normalized_laplacian(adj)
    n_comp = connected_components(adj, directed=False)[0]
    n_req = min(n_comp + k, adj.shape[0] - 1)        # enough modes to clear the null space
    mu, V = eigsh(S, k=n_req, which="LA")            # largest algebraic eigenvalues of S
    order = np.argsort(mu)[::-1]                      # descending mu  => ascending lambda
    mu, V = mu[order], V[:, order]
    lam = 1.0 - mu
    keep = lam > tol                                 # drop ALL near-zero (component) modes
    V, lam, mu = V[:, keep][:, :k], lam[keep][:k], mu[keep][:k]
    return V, lam, mu, deg


def stack_embedding(V, index=None, prefix="lap_emb_"):
    """Stack the k eigenvectors into a per-node k-dimensional embedding table.

    Row i is node i's graph encoding on the first k non-trivial eigenvectors.
    """
    cols = [f"{prefix}{i + 1}" for i in range(V.shape[1])]
    return pd.DataFrame(V, index=index, columns=cols)

In [ ]:
# --- Tunable parameter --------------------------------------------------
K_EIG = 3          # number of smallest non-trivial eigenvectors to keep
# ------------------------------------------------------------------------

n_components = connected_components(A, directed=False)[0]
if n_components > 1:
    print(f"NOTE: {n_components} connected components. The eigenmap skips all "
          f"{n_components} null-space modes and returns the first {K_EIG} with lambda>0, "
          f"but these localise per-component — aim for 1 component via Section 2.")
else:
    print("Graph is fully connected (1 component).")

# First k non-trivial eigenpairs (mu and deg are kept for the Nystrom step)
V, lam, mu, deg = laplacian_eigenmaps(A, k=K_EIG)
print(f"\nLaplacian eigenvalues (ascending, non-trivial): {np.round(lam, 6)}")

# Stack into a per-node k-dimensional embedding table
house_embedding = stack_embedding(V, index=train_house.index)
print(f"Embedding shape: {house_embedding.shape}")
house_embedding.head()

## 5. Nyström out-of-sample extension to the test houses

Same Nyström formula as `03c`, but the affinity `W(x, x_i)` now uses the **same
inverse-distance weights** as the graph (not binary), so the extension is consistent
with the weighted training kernel:

$$f_k(x) = \frac{1}{\mu_k}\sum_i \frac{W(x, x_i)}{\sqrt{d(x)\,d(x_i)}}\,v_k(i),
\qquad W(x, x_i) = \frac{1}{\text{haversine\_km}(x, x_i)}$$

for the test house's `K_NEIGHBORS` nearest training blocks within `RADIUS_WALK_KM`.
`d(x) = Σ_i W(x, x_i)`, `d(x_i)` is the (weighted) training degree `deg`, and
`v_k`, `μ_k` come from Section 4.

**Isolated test houses** (no training block within the radius) get `NaN` rows and are
counted.

In [ ]:
from scipy.sparse import diags


def nystrom_extend(test_latlon, nn_model, V, mu, deg, k_neighbors, radius_rad):
    """Nyström out-of-sample extension with INVERSE-DISTANCE affinity weights.

    For each new point x:
        f_k(x) = (1/mu_k) * sum_i [ W(x, x_i) / sqrt(d(x) * d(x_i)) ] * v_k(i)
    with W(x, x_i) = 1 / haversine_km(x, x_i) over x's k nearest TRAINING blocks
    within the radius — matching the weighted training graph.

    Returns E (n_test, k) embedding (NaN rows for isolated points) and the isolated mask.
    """
    test_rad = np.radians(np.asarray(test_latlon, dtype=float))
    n_test = test_rad.shape[0]
    n_train = V.shape[0]

    # x's k nearest TRAINING nodes, then keep only those within the radius.
    # (No self-match to exclude: test points are not in the training tree.)
    dist, idx = nn_model.kneighbors(test_rad, n_neighbors=k_neighbors)
    keep = dist <= radius_rad
    rows = np.broadcast_to(np.arange(n_test)[:, None], idx.shape)[keep]
    cols = idx[keep]

    # Inverse-distance weights (km), matching the graph construction
    dist_km = dist[keep] * EARTH_RADIUS_KM
    w = 1.0 / np.maximum(dist_km, 1e-3)
    W_ts = csr_matrix((w, (rows, cols)), shape=(n_test, n_train))

    d_test = np.asarray(W_ts.sum(axis=1)).ravel()
    isolated = d_test == 0

    # Symmetric degree normalisation: P = D_test^{-1/2} W D_train^{-1/2}
    d_test_safe = np.where(isolated, 1.0, d_test)      # avoid 0/0; overwritten to NaN below
    deg_safe = np.where(deg > 0, deg, 1.0)
    Dt_inv_sqrt = diags(1.0 / np.sqrt(d_test_safe))
    Dtr_inv_sqrt = diags(1.0 / np.sqrt(deg_safe))
    P = Dt_inv_sqrt @ W_ts @ Dtr_inv_sqrt

    # f_k = (P @ V)[:, k] / mu_k
    E = (P @ V) / mu[np.newaxis, :]
    E[isolated, :] = np.nan
    return E, isolated

In [ ]:
TEST_HOUSE_PATH = "graph_feature_store/test-house.csv"
test_house = pd.read_csv(TEST_HOUSE_PATH)

# Reuse the training-fitted `nn` tree and the Section 4 spectrum (V, mu, deg)
E_test, iso_test = nystrom_extend(
    test_house[["LATITUDE", "LONGITUDE"]].to_numpy(),
    nn, V, mu, deg,
    k_neighbors=K_NEIGHBORS,
    radius_rad=radius_rad,
)

test_embedding = stack_embedding(E_test, index=test_house.index)
print(f"Test houses : {len(test_house):,}")
print(f"Isolated (no training neighbour within radius): "
      f"{int(iso_test.sum()):,} ({iso_test.mean():.2%})")
test_embedding.head()

In [ ]:
# Sanity check: the first embedding axis should vary smoothly across space for BOTH
# the training nodes (V) and the Nystrom-extended test nodes. If the test colours form
# the same gradient as the training colours, the extension is behaving correctly.
train_plot = pd.DataFrame({
    "LONGITUDE": train_house["LONGITUDE"].to_numpy(),
    "LATITUDE": train_house["LATITUDE"].to_numpy(),
    "emb1": V[:, 0],
    "set": "train",
})
test_plot = pd.DataFrame({
    "LONGITUDE": test_house["LONGITUDE"].to_numpy(),
    "LATITUDE": test_house["LATITUDE"].to_numpy(),
    "emb1": E_test[:, 0],
    "set": "test",
})
combined = pd.concat([train_plot, test_plot], ignore_index=True).dropna(subset=["emb1"])

fig = px.scatter(
    combined,
    x="LONGITUDE",
    y="LATITUDE",
    color="emb1",
    symbol="set",
    color_continuous_scale="Viridis",
    opacity=0.7,
    title="Embedding axis 1: training (V) + Nystrom-extended test",
    labels={"LONGITUDE": "Longitude", "LATITUDE": "Latitude", "emb1": "lap_emb_1"},
)
fig.update_traces(marker=dict(size=5))
fig.update_xaxes(range=list(SG_LON_RANGE))
fig.update_yaxes(range=list(SG_LAT_RANGE))
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(width=950, height=650)
fig.show()

## 6. Basic evaluation — information value of the spatial embedding

The graph nodes are the **HDB blocks** (`sg-hdb-block.csv`), and `house_embedding` is
the per-block Laplacian eigenmap. We:

1. Attach the per-block embedding to its composite key `[TOWN, BLOCK, STREET]`.
2. Join it onto the actual rental records (`train-house.csv` / `test-house.csv`) via
   that key — every block is in the graph, so no Nyström is needed here.
3. Fit a plain linear regression of `MONTHLY_RENT` on the embedding and score it on
   the test set, against two baselines: the global mean and raw `LATITUDE/LONGITUDE`.

This isolates the **spatial** information value: the embedding (like raw coordinates)
knows only *where* a block is — not flat size, lease, or type — so a modest R² is
expected. The question is whether the weighted, connected eigenmap beats raw
coordinates for a *linear* model, and whether it improves on the degenerate `03c`
embedding now that the manifold is continuous.

> If Section 4 still reports >1 component, connect the graph first (raise
> `K_NEIGHBORS` / `RADIUS_WALK_KM`) — the embedding is only a coherent island-wide
> coordinate once the component count is 1.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

KEY = ["TOWN", "BLOCK", "STREET"]

# Per-block embedding tagged with its composite key (train_house holds the block rows)
blocks_emb = train_house[KEY].copy()
emb_cols = list(house_embedding.columns)
blocks_emb[emb_cols] = house_embedding.to_numpy()


def _normalize_keys(df):
    """Lower/strip the composite-key columns so the join is robust to case/whitespace/dtype."""
    out = df.copy()
    for col in KEY:
        out[col] = out[col].astype(str).str.strip().str.lower()
    return out


blocks_key = _normalize_keys(blocks_emb)
dups = int(blocks_key.duplicated(subset=KEY).sum())
if dups:
    print(f"NOTE: {dups} duplicate block keys — keeping first occurrence.")
    blocks_key = blocks_key.drop_duplicates(subset=KEY)

# Actual rental records (the regression rows), joined to their block embedding
train_rent = pd.read_csv("graph_feature_store/train-house.csv")
test_rent = pd.read_csv("graph_feature_store/test-house.csv")

train_j = _normalize_keys(train_rent).merge(blocks_key, on=KEY, how="left")
test_j = _normalize_keys(test_rent).merge(blocks_key, on=KEY, how="left")

for name, df in [("train", train_j), ("test", test_j)]:
    miss = int(df[emb_cols[0]].isna().sum())
    print(f"{name}: {len(df):,} rows, {miss:,} unmatched to a block ({miss / len(df):.2%})")

In [ ]:
# Keep only rows that matched a block embedding
train_j = train_j.dropna(subset=emb_cols)
test_j = test_j.dropna(subset=emb_cols)

y_train = train_j["MONTHLY_RENT"].to_numpy()
y_test = test_j["MONTHLY_RENT"].to_numpy()


def score(pred, name):
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print(f"{name:32s} RMSE={rmse:8.2f}  MAE={mae:8.2f}  R2={r2:7.4f}")


def fit_score(cols, name):
    model = LinearRegression().fit(train_j[cols], y_train)
    score(model.predict(test_j[cols]), name)


print("Test-set performance (target = MONTHLY_RENT):\n")
# Baseline 0: predict the training mean for everyone
score(np.full_like(y_test, y_train.mean(), dtype=float), "mean baseline")
# Baseline 1: raw coordinates
fit_score(["LATITUDE", "LONGITUDE"], "raw lat/long")
# Spatial Laplacian embedding
fit_score(emb_cols, f"laplacian embedding ({len(emb_cols)}d)")